# Engine PINN (Colab-Ready)

This notebook is designed to run directly in **Google Colab**.

It includes:
1. Environment setup (installs + repo path setup)
2. Data loading (CSV if available, otherwise synthetic generation)
3. PINN model creation
4. Two-phase training
5. Evaluation + latent extraction
6. Plot and CSV artifact export


In [ ]:
# --- Robust Colab/bootstrap cell (RUN FIRST) ---
import sys
from pathlib import Path


def _find_engine_pinn_parent() -> Path | None:
    candidates = []

    # 1) Current working dir and parents
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    # 2) Common Colab locations
    candidates.extend([
        Path('/content'),
        Path('/content/drive/MyDrive'),
        Path('/content/drive/MyDrive/Colab Notebooks'),
    ])

    # Direct checks first
    seen = set()
    for root in candidates:
        if str(root) in seen or not root.exists():
            continue
        seen.add(str(root))
        if (root / 'engine_pinn' / '__init__.py').exists():
            return root

    # Bounded recursive search in common roots
    search_roots = [Path('/content'), cwd]
    for sroot in search_roots:
        if not sroot.exists():
            continue
        for init_file in sroot.glob('**/engine_pinn/__init__.py'):
            return init_file.parents[1]

    return None


parent = _find_engine_pinn_parent()
if parent is not None and str(parent) not in sys.path:
    sys.path.insert(0, str(parent))

print('cwd:', Path.cwd())
print('engine_pinn parent found:', parent)
print('sys.path[0]:', sys.path[0])


In [ ]:
# Install dependencies (safe to run multiple times)
# If this fails because file not found, clone your repo first (see README steps).
!pip -q install numpy pandas torch scikit-learn matplotlib seaborn

from pathlib import Path
if Path('engine_pinn/requirements.txt').exists():
    !pip -q install -r engine_pinn/requirements.txt


In [ ]:
# Import check with actionable error message
try:
    from engine_pinn.data.dataset import build_dataloaders, load_or_generate_dataframe
    from engine_pinn.models.pinn import EnginePINN
    from engine_pinn.training.loss import PINNLoss
    from engine_pinn.training.trainer import Trainer
    from engine_pinn.utils.config import PhysicsConfig, TrainConfig
    from engine_pinn.utils.plotting import plot_latent_trends, plot_training_curves
    print('✅ engine_pinn imports successful')
except ModuleNotFoundError as e:
    raise RuntimeError(
        'Could not import engine_pinn.\n\n'
        'Fix steps in Colab:\n'
        '1) Clone repo and cd into it:\n'
        '   !git clone https://github.com/<your-org>/<your-repo>.git\n'
        '   %cd <your-repo>\n'
        '2) Re-run the bootstrap cell.\n'
        '3) Confirm this exists: !ls engine_pinn/__init__.py\n'
    ) from e


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s')
logger = logging.getLogger('engine-pinn-colab')

train_cfg = TrainConfig()
phys_cfg = PhysicsConfig()
set_seed(train_cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## Optional: Use your own CSV from Colab files
If you uploaded a CSV to `/content`, set `custom_csv_path` below. Otherwise synthetic data will be generated.


In [ ]:
# Example: custom_csv_path = Path('/content/engine_data.csv')
custom_csv_path = None

if custom_csv_path is not None:
    train_cfg.data_csv = Path(custom_csv_path)

df = load_or_generate_dataframe(train_cfg.data_csv)
print(df.head())
print('Rows:', len(df))


In [ ]:
bundle = build_dataloaders(
    df,
    batch_size=train_cfg.batch_size,
    test_size=train_cfg.test_size,
    val_size=train_cfg.val_size,
    random_state=train_cfg.seed,
)
print('Train/Val/Test sizes:', len(bundle.train_loader.dataset), len(bundle.val_loader.dataset), len(bundle.test_loader.dataset))


In [ ]:
nox_a_init = max(float(df['NOx'].max()), 1e-3)
model = EnginePINN(
    lhv=phys_cfg.lhv,
    nox_a_init=nox_a_init,
    nox_b_init=phys_cfg.nox_b_init,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
loss_fn = PINNLoss(lambda_phys=train_cfg.lambda_phys, lambda_mono=train_cfg.lambda_mono)
trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=train_cfg.checkpoint_dir,
    patience=train_cfg.patience,
)
model


In [ ]:
history = trainer.fit(
    bundle.train_loader,
    bundle.val_loader,
    epochs_phase1=train_cfg.epochs_phase1,
    epochs_phase2=train_cfg.epochs_phase2,
)
print('Completed epochs:', len(history.train_total))


In [ ]:
def evaluate_and_collect_latents(model, loader, y_scaler, device):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(device)
            x_raw = batch['x_raw'].to(device)
            out = model(x, x_raw)

            y_true = y_scaler.inverse_transform(batch['y'].numpy())
            y_pred = np.hstack([out['bsfc'].cpu().numpy(), out['nox'].cpu().numpy()])
            latent = out['latents'].cpu().numpy()
            x_raw_np = batch['x_raw'].cpu().numpy()

            for i in range(len(x_raw_np)):
                rows.append({
                    'BMEP': x_raw_np[i, 0],
                    'H2_percentage': x_raw_np[i, 1],
                    'Spark_Ignition_Timing': x_raw_np[i, 2],
                    'Lambda': x_raw_np[i, 3],
                    'BSFC_true': y_true[i, 0],
                    'NOx_true': y_true[i, 1],
                    'BSFC_pred': y_pred[i, 0],
                    'NOx_pred': y_pred[i, 1],
                    'L1_FMEP': latent[i, 0],
                    'L2_Indicated_Efficiency': latent[i, 1],
                    'L3_Temp_Potential': latent[i, 2],
                })
    return pd.DataFrame(rows)

results_df = evaluate_and_collect_latents(model, bundle.test_loader, bundle.y_scaler, device)
results_df.head()


In [ ]:
train_cfg.plots_dir.mkdir(parents=True, exist_ok=True)

training_curve_path = train_cfg.plots_dir / 'training_curves.png'
latent_plot_path = train_cfg.plots_dir / 'latent_sensitivity.png'
results_path = train_cfg.plots_dir / 'test_predictions_with_latents.csv'

plot_training_curves({'train_total': history.train_total, 'val_total': history.val_total}, training_curve_path)
plot_latent_trends(results_df, latent_plot_path)
results_df.to_csv(results_path, index=False)

print('Saved:', training_curve_path)
print('Saved:', latent_plot_path)
print('Saved:', results_path)


In [ ]:
results_df[['BSFC_true', 'BSFC_pred', 'NOx_true', 'NOx_pred']].describe().T


## Download artifacts in Colab
If running in Colab, you can download generated files using:

```python
from google.colab import files
files.download(str(training_curve_path))
files.download(str(latent_plot_path))
files.download(str(results_path))
```
